# Fashion CNN — Entraînement MobileNetV2

**Avant de lancer :** zipper le dossier `dataset/` et l'uploader sur Google Drive.
```powershell
# Sur ton PC (PowerShell) :
Compress-Archive -Path dataset -DestinationPath dataset.zip
```
Puis glisser `dataset.zip` dans ton Google Drive (racine).

**Important : lancer les cellules dans l'ordre.**

## 0. GPU check + installs

In [ ]:
import torch
print('GPU disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Modèle GPU :', torch.cuda.get_device_name(0))

!pip install -q onnx onnxruntime

## 1. Montage Google Drive et extraction du dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, pathlib

ZIP_PATH    = '/content/drive/MyDrive/dataset.zip'
EXTRACT_DIR = '/content'

# Si le dataset est déjà extrait, on saute l'extraction
candidate = pathlib.Path(EXTRACT_DIR) / 'dataset'

if candidate.exists() and any(candidate.iterdir()):
    print('Dataset déjà présent, extraction ignorée.')
else:
    print('Extraction du dataset...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print('Extraction terminée.')

# Détection automatique du dossier (dataset/ ou dataset/dataset/)
if candidate.exists():
    subdirs = [d for d in candidate.iterdir() if d.is_dir()]
    if len(subdirs) == 1 and subdirs[0].name == 'dataset':
        DATASET_DIR = str(subdirs[0])
    else:
        DATASET_DIR = str(candidate)
else:
    raise FileNotFoundError('Dossier dataset introuvable. Vérifie le zip.')

classes_found = [d.name for d in pathlib.Path(DATASET_DIR).iterdir() if d.is_dir()]
print(f'DATASET_DIR = {DATASET_DIR}')
print(f'{len(classes_found)} classes trouvées : {classes_found}')

## 2. Analyse du dataset — classes et distribution

In [ ]:
import pathlib
import unicodedata
import matplotlib.pyplot as plt
from collections import defaultdict

MIN_IMAGES_PER_CLASS = 20

def nfc(s):
    return unicodedata.normalize('NFC', s)

# ---------------------------------------------------------------------------
# Hiérarchie : sous-classe (nom du dossier) → classe parent pour l'entraînement
# Les sous-classes avec < MIN_IMAGES_PER_CLASS images sont regroupées dans leur
# classe parent. Les sous-dossiers sont conservés sur disque pour usage futur.
# ---------------------------------------------------------------------------
PARENT_CLASS = {
    # Chaussures détaillées → Chaussures
    'Sandales':          'Chaussures',
    'Claquettes_Tongs':  'Chaussures',
    'Mocassins':         'Chaussures',
    'Bottes':            'Chaussures',
    'Bottines':          'Chaussures',
    'Baskets':           'Chaussures',
    # Accessoires détaillés → Accessoires
    'Bijoux':            'Accessoires',
    'Sac':               'Accessoires',
    'Cravate_Noeud':     'Accessoires',
    'Gants':             'Accessoires',
    'Écharpe_Foulard':   'Accessoires',
    'Chapeau':           'Accessoires',
    'Lunettes':          'Accessoires',
    'Ceinture':          'Accessoires',
    # T-shirts détaillés → T-shirt
    'Maillot':           'T-shirt',
    'Top':               'T-shirt',
    # Autres catégories
    'Combinaison':       'Ensemble',
    'Jogging':           'Pantalon',
}
# Normaliser toutes les clés en NFC pour éviter les problèmes d'encodage
PARENT_CLASS = {nfc(k): v for k, v in PARENT_CLASS.items()}

dataset_path = pathlib.Path(DATASET_DIR)
class_counts = {}
for cls_dir in sorted(dataset_path.iterdir()):
    if cls_dir.is_dir():
        n = len([f for f in cls_dir.iterdir() if f.suffix.lower() in ('.jpg', '.jpeg', '.png', '.webp')])
        class_counts[nfc(cls_dir.name)] = n  # normaliser le nom du dossier

# Construire le mapping : dossier → label utilisé pour l'entraînement
class_mapping = {}
for folder_name, count in class_counts.items():
    if folder_name == 'Autre':
        continue  # toujours ignorée
    if count >= MIN_IMAGES_PER_CLASS:
        class_mapping[folder_name] = folder_name       # assez d'images → classe propre
    elif folder_name in PARENT_CLASS:
        class_mapping[folder_name] = PARENT_CLASS[folder_name]  # trop peu → classe parent

# Stats par label d'entraînement
label_counts     = defaultdict(int)
label_subclasses = defaultdict(list)
for folder_name, label in class_mapping.items():
    label_counts[label] += class_counts[folder_name]
    if folder_name != label:
        label_subclasses[label].append(f'{folder_name}({class_counts[folder_name]})')

ignored = {k: v for k, v in class_counts.items() if k not in class_mapping}

print(f"Classes pour l'entraînement ({len(set(class_mapping.values()))}) :")
for label, count in sorted(label_counts.items(), key=lambda x: -x[1]):
    sub = f'  ← {", ".join(label_subclasses[label])}' if label_subclasses[label] else ''
    print(f'  ✓  {label:30s}: {count} images{sub}')

if ignored:
    print(f'\nDossiers ignorés ({len(ignored)}) :')
    for cls, n in sorted(ignored.items(), key=lambda x: -x[1]):
        print(f'  ✗  {cls:30s}: {n} images')

fig, ax = plt.subplots(figsize=(14, 5))
sorted_labels = sorted(label_counts.items(), key=lambda x: -x[1])
ax.bar([x[0] for x in sorted_labels], [x[1] for x in sorted_labels], color='steelblue')
ax.set_title('Distribution des classes (après fusion)')
ax.set_xlabel('Classe')
ax.set_ylabel("Nombre d'images")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 3. DataLoaders PyTorch avec augmentation

In [ ]:
import torch
import unicodedata
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset, Subset
from sklearn.model_selection import train_test_split
from PIL import Image

IMG_SIZE   = 224
BATCH_SIZE = 32
VAL_RATIO  = 0.2
SEED       = 42

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE + 20, IMG_SIZE + 20)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

VALID_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}

class HierarchicalFashionDataset(Dataset):
    """
    Charge les images depuis les sous-dossiers de `root` et applique `class_mapping`
    pour fusionner les petites classes dans leur classe parent.
    Les noms de dossiers sont normalisés en NFC pour éviter les problèmes d'encodage.
    """
    def __init__(self, root, class_mapping, transform=None):
        self.transform = transform
        self.samples   = []

        # Normaliser les clés du mapping en NFC
        mapping = {unicodedata.normalize('NFC', k): v for k, v in class_mapping.items()}

        all_labels = sorted(set(mapping.values()))
        self.classes      = all_labels
        self.class_to_idx = {c: i for i, c in enumerate(all_labels)}

        root_path = pathlib.Path(root)
        for folder in sorted(root_path.iterdir()):
            if not folder.is_dir():
                continue
            folder_nfc = unicodedata.normalize('NFC', folder.name)
            if folder_nfc not in mapping:
                continue
            label_idx = self.class_to_idx[mapping[folder_nfc]]
            for img_path in sorted(folder.iterdir()):
                if img_path.suffix.lower() in VALID_EXTS:
                    self.samples.append((img_path, label_idx))

        self.targets = [s[1] for s in self.samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label


full_dataset = HierarchicalFashionDataset(DATASET_DIR, class_mapping, transform=train_transforms)
val_dataset  = HierarchicalFashionDataset(DATASET_DIR, class_mapping, transform=val_transforms)

CLASS_NAMES = full_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)
print(f'{NUM_CLASSES} classes : {CLASS_NAMES}')

targets = full_dataset.targets
train_idx, val_idx = train_test_split(
    range(len(full_dataset)), test_size=VAL_RATIO, stratify=targets, random_state=SEED
)

train_loader = DataLoader(Subset(full_dataset, train_idx), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(Subset(val_dataset,  val_idx),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train : {len(train_idx)} images | Val : {len(val_idx)} images')

## 4. Modèle — MobileNetV2 (transfer learning)

In [ ]:
import torch.nn as nn
from torchvision import models

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device :', DEVICE)

model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

for param in model.features.parameters():
    param.requires_grad = False

model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.last_channel, 256),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(256, NUM_CLASSES),
)

model = model.to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Paramètres entraînables : {trainable:,} / {total:,}')

## 5. Phase 1 — Entraînement du classifier (features gelées)

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS_PHASE1 = 10

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS_PHASE1)

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * imgs.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0

print('Phase 1 — classifier uniquement')
print(f'{"Epoch":>6} {"Train Loss":>12} {"Train Acc":>12} {"Val Loss":>12} {"Val Acc":>12}')

for epoch in range(1, EPOCHS_PHASE1 + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
    va_loss, va_acc = eval_epoch(model, val_loader, criterion)
    scheduler.step()
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(va_acc)
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(model.state_dict(), '/content/best_model.pth')
    print(f'{epoch:>6} {tr_loss:>12.4f} {tr_acc:>11.1%} {va_loss:>12.4f} {va_acc:>11.1%}')

print(f'\nMeilleure val acc phase 1 : {best_val_acc:.1%}')

## 6. Phase 2 — Fine-tuning (dégel des 5 derniers blocs)

In [ ]:
EPOCHS_PHASE2 = 15

model.load_state_dict(torch.load('/content/best_model.pth'))

for param in model.features[-5:].parameters():
    param.requires_grad = True

optimizer = optim.Adam([
    {'params': model.features[-5:].parameters(), 'lr': 1e-4},
    {'params': model.classifier.parameters(),    'lr': 5e-4},
])
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS_PHASE2)

print('Phase 2 — fine-tuning')
print(f'{"Epoch":>6} {"Train Loss":>12} {"Train Acc":>12} {"Val Loss":>12} {"Val Acc":>12}')

for epoch in range(1, EPOCHS_PHASE2 + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
    va_loss, va_acc = eval_epoch(model, val_loader, criterion)
    scheduler.step()
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(va_acc)
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(model.state_dict(), '/content/best_model.pth')
    print(f'{epoch:>6} {tr_loss:>12.4f} {tr_acc:>11.1%} {va_loss:>12.4f} {va_acc:>11.1%}')

print(f'\nMeilleure val acc totale : {best_val_acc:.1%}')

## 7. Courbes d'apprentissage

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history['train_loss']) + 1)

ax1.plot(epochs_range, history['train_loss'], label='Train')
ax1.plot(epochs_range, history['val_loss'],   label='Val')
ax1.axvline(EPOCHS_PHASE1 + 0.5, color='gray', linestyle='--', label='Fine-tuning')
ax1.set_title('Loss')
ax1.legend()

ax2.plot(epochs_range, history['train_acc'], label='Train')
ax2.plot(epochs_range, history['val_acc'],   label='Val')
ax2.axvline(EPOCHS_PHASE1 + 0.5, color='gray', linestyle='--', label='Fine-tuning')
ax2.set_title('Accuracy')
ax2.set_ylim(0, 1)
ax2.legend()

plt.tight_layout()
plt.show()

## 8. Rapport de classification détaillé

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

model.load_state_dict(torch.load('/content/best_model.pth'))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(DEVICE)
        preds = model(imgs).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cmap='Blues', ax=ax)
ax.set_xlabel('Prédit')
ax.set_ylabel('Réel')
ax.set_title('Matrice de confusion')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 9. Export ONNX (pour FastAPI)

In [ ]:
import onnx, onnxruntime as ort, json

model.load_state_dict(torch.load('/content/best_model.pth'))
model.eval().cpu()

dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
onnx_path   = '/content/fashion_classifier.onnx'

torch.onnx.export(
    model, dummy_input, onnx_path,
    export_params=True,
    opset_version=17,
    input_names=['image'],
    output_names=['logits'],
    dynamic_axes={'image': {0: 'batch_size'}, 'logits': {0: 'batch_size'}}
)

onnx.checker.check_model(onnx_path)
print('Modèle ONNX valide :', onnx_path)

sess = ort.InferenceSession(onnx_path)
out  = sess.run(None, {'image': dummy_input.numpy()})
print('Sortie ONNX shape :', out[0].shape)

with open('/content/class_names.json', 'w', encoding='utf-8') as f:
    json.dump(CLASS_NAMES, f, ensure_ascii=False)
print('Classes sauvegardées :', CLASS_NAMES)

## 10. Copie des fichiers vers Google Drive

In [ ]:
import shutil, os

DRIVE_OUTPUT = '/content/drive/MyDrive/fashion_model'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

files_to_copy = [
    '/content/best_model.pth',
    '/content/fashion_classifier.onnx',
    '/content/fashion_classifier.onnx.data',  # présent si le modèle est volumineux
    '/content/class_names.json',
]

for src in files_to_copy:
    if os.path.exists(src):
        fname = os.path.basename(src)
        shutil.copy(src, f'{DRIVE_OUTPUT}/{fname}')
        print(f'  ✓ {fname} copié')
    else:
        print(f'  - {os.path.basename(src)} absent (ignoré)')

print('\nFichiers dans Drive :')
for fname in os.listdir(DRIVE_OUTPUT):
    size = os.path.getsize(f'{DRIVE_OUTPUT}/{fname}') / 1024 / 1024
    print(f'  {fname}: {size:.1f} MB')